# 5.3. Stage 5: Daily Full Rejoin

**Stage:** 5.3 of 5

**Purpose:** Create a fully enriched daily full dataset. Unlike Stage 5.1, this stage retains every daily vacancy occurrence. If an ID has no classified Stage 4 record on the same day, other Stage 4 days are searched for the first available classified record.

**Input:**
- Stage 1 daily id/region/click Pickle files
- Stage 4 daily classified Pickle files
- Stage 4.5 region database

**Output:** One daily full Parquet file plus full and Ukraine-country JSON files. The process tracker stores the Parquet path.

Unclassified records are retained with missing Stage 4 enrichment fields.

## 5.3.1. Load configuration and dependencies

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys
import time

import pandas as pd

from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g

g.clean_memory()

## 5.3.2. Build or load the full-data tracker

In [ ]:
def get_stage5_process_df(stage5_process_path, stage1_region_path):
    if os.path.exists(stage5_process_path):
        tracker = pd.read_pickle(stage5_process_path)
    else:
        rows = []
        for filename in sorted(os.listdir(stage1_region_path)):
            file_path = os.path.join(stage1_region_path, filename)
            if not os.path.isfile(file_path):
                continue
            rows.append(
                {
                    "input_path": g.get_file_name_without_ext(filename),
                    "region_path": file_path,
                    "rejoin_path": pd.NA,
                    "rejoin_status": pd.NA,
                }
            )
        tracker = pd.DataFrame(
            rows,
            columns=["input_path", "region_path", "rejoin_path", "rejoin_status"],
        )
        tracker.to_pickle(stage5_process_path)

    return tracker.sort_values("input_path").reset_index(drop=True)


process_df = get_stage5_process_df(
    g.Config.STAGE5_PROCESS_FULL_PATH,
    g.Config.STAGE1_ID_REGION_NCLICK_PATH,
)

g.check_folder_exists(g.Config.STAGE5_DAILY_FULL_PARQUET_PATH)
g.check_folder_exists(g.Config.STAGE5_DAILY_FULL_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_DAILY_FULL_JSON_UA_PATH)

## 5.3.3. Prepare reference data and output schema

In [ ]:
region_final_db = pd.read_pickle(g.Config.STAGE4_5_REGION_DB_PATH)
region_final_db = region_final_db.drop_duplicates(subset="original", keep="first")

output_columns = [
    "id", "title", "description", "min_salary", "max_salary", "currency",
    "salary_rate", "date_created", "date_expired", "clean_title", "clean_desc",
    "title_lang", "desc_lang", "skill_ids", "skill_labels", "job_type",
    "classified_code", "classified_title", "skill_labels_en",
    "classified_title_clean", "extract_type", "esco_title", "esco_skills",
    "esco_id", "esco_code", "number_of_clicks", "date", "region_original",
    "city", "district", "region", "country", "latitude", "longitude",
]


def read_stage4_file(path):
    try:
        data = pd.read_pickle(path)
    except Exception as error:
        print(f"Cannot read Stage 4 file {path}: {error}")
        return None

    if data.empty or "id" not in data.columns:
        print(f"Skipping unusable Stage 4 file: {path}")
        return None

    data = data.copy()
    data["id"] = data["id"].astype("string")
    return data.drop_duplicates(subset="id", keep="first")


stage4_sources = []
for _, tracker_row in process_df.iterrows():
    stage4_path = os.path.join(
        g.Config.STAGE4_OUTPUT_PATH,
        tracker_row["input_path"] + ".pkl",
    )
    if os.path.isfile(stage4_path):
        stage4_sources.append((tracker_row["input_path"], stage4_path))

print(f"Available Stage 4 daily files: {len(stage4_sources)}")

## 5.3.4. Create daily full Parquet files

For each Stage 1 day, same-day Stage 4 data is searched first, followed by the remaining Stage 4 files. Classified matches take priority; if no classified match exists, the first available unclassified Stage 4 record is retained as a fallback. All original Stage 1 rows remain in the output.

In [ ]:
for process_index, process_row in process_df.iterrows():
    started_at = time.time()
    input_name = process_row["input_path"]
    parquet_path = os.path.join(
        g.Config.STAGE5_DAILY_FULL_PARQUET_PATH,
        input_name + ".parquet",
    )
    json_path = os.path.join(
        g.Config.STAGE5_DAILY_FULL_JSON_PATH,
        input_name + ".json",
    )
    ua_json_path = os.path.join(
        g.Config.STAGE5_DAILY_FULL_JSON_UA_PATH,
        input_name + ".json",
    )

    if os.path.isfile(parquet_path):
        process_df.loc[process_index, "rejoin_path"] = parquet_path
        process_df.loc[process_index, "rejoin_status"] = "complete"
        process_df.to_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)
        print(f"{input_name}: existing Parquet registered; skipped.")
        continue

    region_path = process_row["region_path"]
    if pd.isna(region_path) or not os.path.isfile(region_path):
        print(f"{input_name}: Stage 1 region file is missing; skipped.")
        continue

    try:
        base_data = pd.read_pickle(region_path)
    except Exception as error:
        print(f"{input_name}: cannot read Stage 1 region file: {error}")
        continue

    if base_data.empty or "id" not in base_data.columns:
        print(f"{input_name}: Stage 1 region file is empty or has no id; skipped.")
        continue

    base_data = base_data.copy()
    base_data["id"] = base_data["id"].astype("string")
    base_data["_row_order"] = range(base_data.shape[0])
    base_data = base_data.merge(
        region_final_db,
        how="left",
        left_on="region",
        right_on="original",
        validate="many_to_one",
    )
    if "region_x" in base_data.columns:
        base_data = base_data.drop(columns="region_x")
    base_data = base_data.rename(
        columns={"original": "region_original", "region_y": "region"}
    )

    base_columns = set(base_data.columns)
    target_ids = set(base_data["id"].dropna().tolist())
    unresolved_ids = set(target_ids)
    classified_parts = []
    fallback_parts = []
    fallback_ids = set()

    prioritized_sources = sorted(
        stage4_sources,
        key=lambda item: (item[0] != input_name, item[0]),
    )

    for source_name, stage4_path in prioritized_sources:
        if not unresolved_ids and fallback_ids.issuperset(target_ids):
            break

        stage4_data = read_stage4_file(stage4_path)
        if stage4_data is None:
            continue

        payload_columns = [
            column for column in stage4_data.columns
            if column == "id" or column not in base_columns
        ]
        stage4_payload = stage4_data[payload_columns]
        relevant = stage4_payload[stage4_payload["id"].isin(target_ids)]

        new_fallback = relevant[~relevant["id"].isin(fallback_ids)]
        if not new_fallback.empty:
            fallback_parts.append(new_fallback)
            fallback_ids.update(new_fallback["id"].dropna().tolist())

        if "esco_title" not in relevant.columns:
            continue
        classified = relevant[
            relevant["id"].isin(unresolved_ids)
            & relevant["esco_title"].notna()
        ]
        if classified.empty:
            continue

        classified_parts.append(classified)
        unresolved_ids.difference_update(classified["id"].dropna().tolist())

    if classified_parts:
        classified_lookup = pd.concat(classified_parts, ignore_index=True)
        classified_lookup = classified_lookup.drop_duplicates(subset="id", keep="first")
    else:
        classified_lookup = pd.DataFrame(columns=["id"])

    if fallback_parts:
        fallback_lookup = pd.concat(fallback_parts, ignore_index=True)
        fallback_lookup = fallback_lookup.drop_duplicates(subset="id", keep="first")
        fallback_lookup = fallback_lookup[
            ~fallback_lookup["id"].isin(classified_lookup["id"])
        ]
    else:
        fallback_lookup = pd.DataFrame(columns=["id"])

    enrichment_lookup = pd.concat(
        [classified_lookup, fallback_lookup],
        ignore_index=True,
        sort=False,
    ).drop_duplicates(subset="id", keep="first")

    daily_full = base_data.merge(
        enrichment_lookup,
        on="id",
        how="left",
        validate="many_to_one",
    )
    daily_full = daily_full.sort_values("_row_order").drop(columns="_row_order")
    daily_full = daily_full.reindex(columns=output_columns).reset_index(drop=True)

    daily_full.to_parquet(parquet_path, index=False)
    daily_full.to_json(json_path, orient="records", date_format="iso")
    daily_full[daily_full["country"] == "Ukraine"].to_json(
        ua_json_path,
        orient="records",
        force_ascii=False,
        date_format="iso",
    )

    process_df.loc[process_index, "rejoin_path"] = parquet_path
    process_df.loc[process_index, "rejoin_status"] = "complete"
    process_df.to_pickle(g.Config.STAGE5_PROCESS_FULL_PATH)

    unresolved_count = daily_full["esco_title"].isna().sum()
    elapsed = time.time() - started_at
    print(
        f"{input_name}: complete — {daily_full.shape[0]} rows, "
        f"{unresolved_count} without ESCO title, {elapsed:.2f} s."
    )

print("All done!")

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.